In [1]:
from pathlib import Path
import logging, sys
import math
from shapely.geometry import box, Point
from pyproj import Transformer

# Logger
def setup_logger(name="aoi", level=logging.INFO):
    log = logging.getLogger(name); log.setLevel(level)
    if not log.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
        log.addHandler(h)
    return log

log = setup_logger()

# --- 1) Conversion de coordonnées vers EPSG:2154 (Lambert-93)
def to_2154_from_lonlat(lon, lat):
    tf = Transformer.from_crs("EPSG:4326", "EPSG:2154", always_xy=True)
    x, y = tf.transform(lon, lat)
    return x, y

def to_2154(x, y, crs_in="EPSG:2154"):
    if crs_in == "EPSG:2154":
        return x, y
    tf = Transformer.from_crs(crs_in, "EPSG:2154", always_xy=True)
    return tf.transform(x, y)

# --- 2) Bbox centrée sur un point
def point_bbox_around(x2154, y2154, half_size_m):
    """Carré centré (minx,miny,maxx,maxy)"""
    return (x2154 - half_size_m, y2154 - half_size_m,
            x2154 + half_size_m, y2154 + half_size_m)

# --- 3) Buffer d’ombre
def shadow_buffer_m(Hmax_m, hmin_deg):
    return Hmax_m / math.tan(math.radians(hmin_deg))

# --- 4) AAOI = AOI centrée + buffer d’ombre
def aaoI_from_point(x, y, base_halfsize_m, Hmax_m, hmin_deg, crs_in="EPSG:2154"):
    X, Y = to_2154(x, y, crs_in=crs_in)
    base_bbox = point_bbox_around(X, Y, base_halfsize_m)
    B = shadow_buffer_m(Hmax_m, hmin_deg)
    minx, miny, maxx, maxy = base_bbox
    aaoI = (minx - B, miny - B, maxx + B, maxy + B)
    return aaoI, B

# --- 5) Exemple d’usage
# Cas 1: Latitude : 46.201238 | Longitude : 6.32095
lon, lat = 6.32095, 46.201238
# Fenêtre de base 5 km de demi-largeur, relief bloquant 1000 m, h_min=10°
AAOI_bounds, Bm = aaoI_from_point(lon, lat, base_halfsize_m=5000, Hmax_m=1000, hmin_deg=10, crs_in="EPSG:4326")
log.info(f"AAOI bbox (EPSG:2154): {AAOI_bounds} | buffer ombre B={Bm/1000:.1f} km")

# Cas 2: point déjà en EPSG:2154
# x2154, y2154 = 950000, 6270000
# AAOI_bounds, Bm = aaoI_from_point(x2154, y2154, 5000, 2500, 5, crs_in='EPSG:2154')


[INFO] AAOI bbox (EPSG:2154): (945331.4633142418, 6561534.405115804, 966674.0269534773, 6582876.968755038) | buffer ombre B=5.7 km
